In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
PIPELINE_NAME = "Gold Monthly Revenue"
SOURCE_TABLE = GOLD_FACT_SALES
TARGET_TABLE = GOLD_MONTHLY_REVENUE
RUN_ID = generate_run_id()
START_TIME = datetime.now()

In [0]:
print("GOLD MONTHLY REVENUE PIPELINE")

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID : {RUN_ID}")
print(f"Target : {TARGET_TABLE}")

GOLD MONTHLY REVENUE PIPELINE
Pipeline : Gold Monthly Revenue
Run ID : 09fdb0ab-2633-4db4-b4e1-315dfad72195
Target : retailmart.gold.monthly_revenue


In [0]:
fact_sales_df = spark.table(SOURCE_TABLE)
display(fact_sales_df.limit(10))

order_id,order_item_id,order_status,order_purchase_timestamp,order_year,order_month,revenue_month,order_delivered_customer_date,delivery_duration_days,customer_id,customer_city,customer_state,product_id,product_category_name,price,freight_value,total_item_value,payment_type,payment_installments,total_payment_value
ORD_0000001,1,delivered,2023-06-15T14:30:00.000Z,2023,6,2023-06,2023-06-19T14:30:00.000Z,4,CUST_006571,Belo Horizonte,MG,PROD_001950,books,1754.7,20.17,1774.87,multiple,12,2807.85
ORD_0000002,1,cancelled,2021-06-12T11:02:00.000Z,2021,6,2021-06,null,null,CUST_006956,Aracaju,SE,PROD_000989,food,2105.2,45.48,2150.68,voucher,1,805.73
ORD_0000003,1,delivered,2022-03-31T19:38:00.000Z,2022,3,2022-03,2022-04-14T19:38:00.000Z,14,CUST_008373,Joao Pessoa,PB,PROD_000254,furniture,1627.94,78.29,1706.23,credit_card,6,508.88
ORD_0000004,1,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001327,music,65.22,31.08,96.3,boleto,3,1624.71
ORD_0000004,2,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_002714,computers,946.39,44.43,990.82,boleto,3,1624.71
ORD_0000004,3,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001507,home_appliances,157.96,55.93,213.89,boleto,3,1624.71
ORD_0000005,1,shipped,2022-08-27T07:46:00.000Z,2022,8,2022-08,null,null,CUST_002810,Manaus,AM,PROD_001872,fashion,1681.83,14.0,1695.83,credit_card,3,1592.91
ORD_0000006,1,invoiced,2023-05-26T16:27:00.000Z,2023,5,2023-05,null,null,CUST_007867,Recife,PE,PROD_001518,garden,856.74,64.83,921.57,credit_card,1,495.97
ORD_0000007,1,processing,2021-09-21T15:50:00.000Z,2021,9,2021-09,null,null,CUST_004827,Campo Grande,MS,PROD_001504,music,373.8,66.88,440.68,credit_card,2,1738.92
ORD_0000008,1,delivered,2021-01-19T14:24:00.000Z,2021,1,2021-01,2021-01-28T14:24:00.000Z,9,CUST_005137,Porto Velho,RO,PROD_001133,health,899.98,23.95,923.93,voucher,1,1107.69


In [0]:
print(f"Total Records : {fact_sales_df.count()}")
fact_sales_df.printSchema()

Total Records : 86328
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_month: string (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- total_payment_value: double (nullable = true)



In [0]:
# the joins were performed in the fact_sales

In [0]:
%sql
CREATE OR REPLACE TABLE retailmart.gold.monthly_revenue AS

SELECT
    DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS revenue_month,
    YEAR(order_purchase_timestamp) AS order_year,
    MONTH(order_purchase_timestamp) AS order_month,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_item_value) AS monthly_revenue,
    AVG(total_item_value) AS average_item_value
FROM retailmart.gold.fact_sales
WHERE order_status = 'delivered'
GROUP BY
    DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM'),
    YEAR(order_purchase_timestamp),
    MONTH(order_purchase_timestamp)
ORDER BY
    order_year,
    order_month;

num_affected_rows,num_inserted_rows


In [0]:
monthly_revenue_df = spark.table(TARGET_TABLE)
display(monthly_revenue_df)

revenue_month,order_year,order_month,total_orders,monthly_revenue,average_item_value
2021-01,2021,1,865,1859832.1499999978,1269.5099999999984
2021-02,2021,2,733,1736562.1899999997,1310.6129735849054
2021-03,2021,3,886,1989130.3599999985,1267.7695092415543
2021-04,2021,4,853,1861818.3400000029,1301.061034241791
2021-05,2021,5,856,1898780.9299999983,1326.890936408105
2021-06,2021,6,821,1830603.0100000002,1303.848297720798
2021-07,2021,7,867,2055580.0899999987,1321.0668958868887
2021-08,2021,8,834,1881795.5199999998,1296.8956030323914
2021-09,2021,9,838,1903691.0000000002,1306.582704186685
2021-10,2021,10,842,1845287.189999998,1271.7347966919353


In [0]:
display(
    monthly_revenue_df.orderBy("order_year","order_month")
)

revenue_month,order_year,order_month,total_orders,monthly_revenue,average_item_value
2021-01,2021,1,865,1859832.1499999978,1269.5099999999984
2021-02,2021,2,733,1736562.1899999997,1310.6129735849054
2021-03,2021,3,886,1989130.3599999985,1267.7695092415543
2021-04,2021,4,853,1861818.3400000029,1301.061034241791
2021-05,2021,5,856,1898780.9299999983,1326.890936408105
2021-06,2021,6,821,1830603.0100000002,1303.848297720798
2021-07,2021,7,867,2055580.0899999987,1321.0668958868887
2021-08,2021,8,834,1881795.5199999998,1296.8956030323914
2021-09,2021,9,838,1903691.0000000002,1306.582704186685
2021-10,2021,10,842,1845287.189999998,1271.7347966919353


In [0]:
rows_written = monthly_revenue_df.count()
print(f"Rows Written : {rows_written}")

Rows Written : 30


In [0]:
assert rows_written > 0, "Monthly Revenue table is empty!"

In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_TABLE,
    target=TARGET_TABLE,
    rows_read=fact_sales_df.count(),
    rows_written=rows_written,
    duplicate_count=0,
    start_time=START_TIME,
    status="SUCCESS"
)

LOAD REPORT
Pipeline        : Gold Monthly Revenue
Run ID          : 09fdb0ab-2633-4db4-b4e1-315dfad72195
Source          : retailmart.gold.fact_sales
Target          : retailmart.gold.monthly_revenue
Rows Read       : 86328
Rows Written    : 30
Duplicate Rows  : 0
Start Time      : 2026-07-19 04:53:59.673369
End Time        : 2026-07-19 04:54:10.376984
Duration (sec)  : 10.7
Status          : SUCCESS


## Engineering Observations

- The Monthly Revenue Gold table summarizes business revenue on a monthly basis using validated transaction data from the Gold Fact Sales table.

- Revenue calculations consider only delivered orders using the WHERE clause, ensuring that cancelled or unavailable orders are excluded from financial reporting.

- SQL GROUP BY was used to aggregate revenue and order counts at the Year-Month level, producing business-ready KPIs for trend analysis.

- Average Item Value (AIV) was also calculated to support customer spending analysis and executive reporting.

- The resulting dataset is optimized for dashboards, time-series visualizations, and business intelligence reporting in Power BI.

- By reusing the Gold Fact Sales table, duplicate transformation logic is avoided while maintaining a scalable Medallion Architecture.